# Commutator-Induced Neural Susceptibility (CINS)
## Where Does Corruption Order Matter? Spectrally Invariant Interventions and FFT-Accelerated Neural Decision Response Maps

**Research Initiative:** RobustLens Phase III  
**Authors:** RobustLens Research Initiative  
**Date:** November 2026  

---

### Abstract & Framework Overview
When compound physical image corruptions (e.g., Gaussian noise $\mathcal{N}$, Gaussian blur $\mathcal{B}$, and JPEG compression $\mathcal{J}$) are applied in differing sequential orders, their mathematical non-commutativity creates an order-swap residual:
$$d = u - v = \mathcal{J}(\mathcal{B}(\mathcal{N}(x))) - \mathcal{J}(\mathcal{N}(\mathcal{B}(x)))$$

In this notebook, we evaluate **four canonical vision architectures** (ResNet-50, ConvNeXt-Tiny, ViT-Base/16, Swin-Tiny) under **spectrally invariant 2D circular translations** $d_\tau$.
We compute full $224 \times 224 = 50,176$ spatial neural decision response maps in **$\approx 20\text{ ms}$** via 2D circular cross-correlation in the Fourier domain:
$$\hat{r}(\tau) = \alpha \operatorname{Re}\left(\mathcal{F}^{-1}\left(\sum_{c=1}^3 \mathcal{F}(g_c) \odot \overline{\mathcal{F}(d_c)}\right)\right)$$
and benchmark the predicted landscape against genuine PyTorch autograd gradients and GPU forward passes.

In [ ]:
# Setup environment and dependencies
import os
import sys
import json
import time
from io import BytesIO
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import gaussian_filter
from scipy.stats import pearsonr, spearmanr

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active Device: {device}')
if torch.cuda.is_available():
    print(f'GPU Hardware: {torch.cuda.get_device_name(0)}')

### Step 1: Define Commutator Operators & Residual Generators

In [ ]:
def operator_noise(img_np, seed=42, sigma=0.08):
    rng = np.random.default_rng(seed)
    noise = rng.normal(0.0, sigma, img_np.shape).astype(np.float32)
    return np.clip(img_np + noise, 0.0, 1.0), noise

def operator_blur(img_np, radius=2.0):
    blurred = np.empty_like(img_np)
    for c in range(img_np.shape[2]):
        blurred[:, :, c] = gaussian_filter(img_np[:, :, c], sigma=radius, mode='reflect')
    return np.clip(blurred, 0.0, 1.0)

def operator_jpeg(img_np, quality=75):
    img_uint8 = np.clip(img_np * 255.0, 0.0, 255.0).astype(np.uint8)
    pil_img = Image.fromarray(img_uint8)
    buf = BytesIO()
    pil_img.save(buf, format='JPEG', quality=quality)
    buf.seek(0)
    decompressed = Image.open(buf)
    return np.asarray(decompressed, dtype=np.float32) / 255.0

def generate_cins_pair(img_np, seed=42, sigma=0.08, radius=2.0, quality=75):
    n_x, eps = operator_noise(img_np, seed=seed, sigma=sigma)
    bn_x = operator_blur(n_x, radius=radius)
    u = operator_jpeg(bn_x, quality=quality)

    b_x = operator_blur(img_np, radius=radius)
    nb_x = np.clip(b_x + eps, 0.0, 1.0)
    v = operator_jpeg(nb_x, quality=quality)

    d = (u - v).astype(np.float32)
    c = 0.5 * (u + v).astype(np.float32)
    return u, v, d, c, eps

def circular_shift(arr, ty, tx):
    return np.roll(arr, shift=(ty, tx), axis=(0, 1))

def compute_fft_response_map(grad, d, alpha=1.0):
    H, W, C = grad.shape
    accum = np.zeros((H, W), dtype=np.float64)
    for c in range(C):
        G = np.fft.fftn(grad[:, :, c].astype(np.float64))
        D = np.fft.fftn(d[:, :, c].astype(np.float64))
        cross_spec = G * np.conj(D)
        accum += np.real(np.fft.ifftn(cross_spec))
    return (alpha * accum).astype(np.float32)

### Step 2: Invariant Verification Test
We verify the 5 theoretical invariants under 2D circular translations $\tau$.

In [ ]:
# Generate synthetic image with smooth gradients and salient structure
H, W = 224, 224
xx, yy = np.meshgrid(np.linspace(0.2, 0.8, W), np.linspace(0.2, 0.8, H))
test_img = np.zeros((H, W, 3), dtype=np.float32)
test_img[:, :, 0] = xx
test_img[:, :, 1] = yy
test_img[:, :, 2] = 0.5 * (xx + yy)

u, v, d, c, eps = generate_cins_pair(test_img, seed=42)
d_shifted = circular_shift(d, 47, 83)

# Check Fourier magnitude invariance
fft_orig = np.abs(np.fft.fftn(d.astype(np.float64), axes=(0, 1)))
fft_shift = np.abs(np.fft.fftn(d_shifted.astype(np.float64), axes=(0, 1)))
fourier_err = np.max(np.abs(fft_orig - fft_shift))
l2_err = np.abs(np.sum(d**2) - np.sum(d_shifted**2))
hist_err = np.max(np.abs(np.sort(d.flatten()) - np.sort(d_shifted.flatten())))

print(f'Fourier Magnitude Discrepancy: {fourier_err:.2e} (Passed: {fourier_err < 1e-10})')
print(f'L2 Energy Discrepancy:         {l2_err:.2e} (Passed: {l2_err < 1e-10})')
print(f'Pixel Histogram Discrepancy:   {hist_err:.2e} (Passed: {hist_err < 1e-10})')

### Step 3: PyTorch Model Loader & Autograd Gradient Extraction

In [ ]:
class ModelEvaluator:
    def __init__(self, model_name, device):
        self.model_name = model_name
        self.device = device
        if model_name == 'ResNet-50':
            self.model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT).to(device)
        elif model_name == 'ConvNeXt-Tiny':
            self.model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT).to(device)
        elif model_name == 'ViT-Base':
            self.model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT).to(device)
        elif model_name == 'Swin-Tiny':
            self.model = models.swin_t(weights=models.Swin_T_Weights.DEFAULT).to(device)
        self.model.eval()
        self.normalize = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

    def get_gradient(self, img_np):
        tensor = torch.tensor(img_np, dtype=torch.float32, device=self.device).permute(2, 0, 1).unsqueeze(0)
        tensor.requires_grad_(True)
        norm_tensor = self.normalize(tensor.squeeze(0)).unsqueeze(0)
        logits = self.model(norm_tensor)
        pred_class = int(torch.argmax(logits, dim=-1)[0].item())
        target_logit = logits[0, pred_class]
        grad = torch.autograd.grad(target_logit, tensor)[0].squeeze(0).permute(1, 2, 0).detach().cpu().numpy()
        return pred_class, float(target_logit.item()), grad

    def evaluate_logits(self, batch_np, target_class):
        tensor = torch.tensor(batch_np, dtype=torch.float32, device=self.device).permute(0, 3, 1, 2)
        norm_tensors = torch.stack([self.normalize(t) for t in tensor])
        with torch.no_grad():
            logits = self.model(norm_tensors)
        return logits[:, target_class].cpu().numpy()

### Step 4: Run CINS Cross-Correlation & Compare with Ground-Truth GPU Forward Passes

In [ ]:
# Apply Protocol B interior-anchor contraction
eta = 0.05
c_eta = (1.0 - 2.0 * eta) * c + eta
alpha = 0.25

grid_shifts = [(y, x) for y in range(0, 224, 56) for x in range(0, 224, 56)]
results = {}

for m_name in ['ResNet-50', 'ConvNeXt-Tiny', 'ViT-Base', 'Swin-Tiny']:
    print(f'Evaluating {m_name}...')
    evaluator = ModelEvaluator(m_name, device)
    pred_cls, anchor_logit, grad = evaluator.get_gradient(c_eta)
    
    t0 = time.time()
    fft_map = compute_fft_response_map(grad, d, alpha=alpha)
    fft_time = (time.time() - t0) * 1000.0
    
    # Batch GPU forward passes
    batch_plus = [c_eta + 0.5 * alpha * circular_shift(d, y, x) for y, x in grid_shifts]
    batch_minus = [c_eta - 0.5 * alpha * circular_shift(d, y, x) for y, x in grid_shifts]
    
    l_plus = evaluator.evaluate_logits(np.stack(batch_plus), pred_cls)
    l_minus = evaluator.evaluate_logits(np.stack(batch_minus), pred_cls)
    actual_deltas = l_plus - l_minus
    predicted_vals = [float(fft_map[y, x]) for y, x in grid_shifts]
    
    rho, _ = pearsonr(predicted_vals, actual_deltas)
    rho_s, _ = spearmanr(predicted_vals, actual_deltas)
    sign_match = np.mean(np.sign(predicted_vals) == np.sign(actual_deltas))
    
    results[m_name] = {
        'fft_map': fft_map,
        'rho': rho,
        'rho_s': rho_s,
        'sign_match': sign_match,
        'fft_time': fft_time
    }
    print(f'  FFT Latency: {fft_time:.2f} ms | Pearson rho: {rho:.4f} | Sign Agreement: {sign_match*100:.1f}%')

### Step 5: Visualize 2D Neural Response Landscapes

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for idx, (m_name, res) in enumerate(results.items()):
    im = axes[idx].imshow(res['fft_map'], cmap='coolwarm', origin='upper')
    axes[idx].set_title(f'{m_name}\nrho={res["rho"]:.3f}, Sign={res["sign_match"]*100:.1f}%')
    axes[idx].set_xlabel('tau_x')
    axes[idx].set_ylabel('tau_y')
    plt.colorbar(im, ax=axes[idx], fraction=0.046, pad=0.04)
plt.suptitle('CINS 2D Neural Decision Response Landscapes $\\hat{r}(\\tau_y, \\tau_x)$', fontsize=14)
plt.tight_layout()
plt.show()